# Etapa 2: Limpieza y Transformación de Datos (Data Cleaning)

Este notebook aplica las reglas de limpieza definidas en el diagnóstico para preparar el dataset analítico.

In [28]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import time

warnings.filterwarnings('ignore')
start_time = time.time()
print("Librerías cargadas.")

Librerías cargadas.


In [29]:
print("1. Cargando datos crudos...")
df = pd.read_parquet('../datos/raw/yellow_tripdata_2025-01.parquet')
original_len = len(df)
print(f"Filas originales: {original_len:,}")

1. Cargando datos crudos...
Filas originales: 3,475,226


In [30]:
print("2. Filtrado temporal estricto (Enero 2025)...")
df = df[(df['tpep_pickup_datetime'] >= '2025-01-01') & (df['tpep_pickup_datetime'] < '2025-02-01')]
print(f"Filas retenidas después de filtrado temporal: {len(df):,} ({(len(df)/original_len)*100:.2f}%)")

2. Filtrado temporal estricto (Enero 2025)...
Filas retenidas después de filtrado temporal: 3,475,204 (100.00%)


In [31]:
print("3. Tratamiento de Valores Nulos...")

# Imputar moda
df['passenger_count'] = df['passenger_count'].fillna(1.0)
df['RatecodeID'] = df['RatecodeID'].fillna(1.0)

# Imputar 0 a cargos no registrados
df['congestion_surcharge'] = df['congestion_surcharge'].fillna(0.0)
df['Airport_fee'] = df['Airport_fee'].fillna(0.0)

print(f"Nulos restantes en columnas clave: {df[['passenger_count', 'RatecodeID', 'congestion_surcharge', 'Airport_fee']].isnull().sum().sum()}")

3. Tratamiento de Valores Nulos...
Nulos restantes en columnas clave: 0


In [32]:
print("4. Limpieza de Outliers (Anomalías Lógicas)...")

# Tarifas: mayores que 0 y razonables (< 500)
df = df[(df['fare_amount'] > 0) & (df['fare_amount'] <= 500)]

# Distancias: mayores que 0 y razonables (< 100)
df = df[(df['trip_distance'] > 0) & (df['trip_distance'] <= 100)]

print(f"Filas retenidas después de limpieza de outliers: {len(df):,} ({(len(df)/original_len)*100:.2f}%)")

4. Limpieza de Outliers (Anomalías Lógicas)...
Filas retenidas después de limpieza de outliers: 3,253,682 (93.63%)


In [33]:
print("5. Ingeniería de Características (Feature Engineering)...")

# Duración del viaje
df['duracion_minutos'] = (df['tpep_dropoff_datetime'] - df['tpep_pickup_datetime']).dt.total_seconds() / 60.0

# Limpiar anomalías de duración
df = df[(df['duracion_minutos'] > 0) & (df['duracion_minutos'] <= 180)]

# Velocidad media (mph)
df['velocidad_mph'] = df['trip_distance'] / (df['duracion_minutos'] / 60.0)

# Limpiar anomalías de velocidad (mayores a 80 mph en ciudad es poco realista)
df = df[df['velocidad_mph'] <= 80]

# Variables temporales extraídas
df['dia_semana'] = df['tpep_pickup_datetime'].dt.day_name()
df['hora_dia'] = df['tpep_pickup_datetime'].dt.hour
df['es_fin_de_semana'] = df['dia_semana'].isin(['Saturday', 'Sunday']).astype(int)

print(f"Filas retenidas después de Feature Engineering: {len(df):,} ({(len(df)/original_len)*100:.2f}%)")

5. Ingeniería de Características (Feature Engineering)...
Filas retenidas después de Feature Engineering: 3,250,191 (93.52%)


In [34]:
print("6. Guardando el dataset analítico limpio...")
out_path = '../datos/procesados/yellow_tripdata_2025-01_clean.parquet'
df.to_parquet(out_path, index=False)

end_time = time.time()
print(f"Proceso completado en {(end_time - start_time)/60:.2f} minutos.")
print(f"Dataset guardado en: {out_path}")
print(f"Retención de datos final: {(len(df)/original_len)*100:.2f}% ({len(df):,} filas validas).")

6. Guardando el dataset analítico limpio...
Proceso completado en 0.06 minutos.
Dataset guardado en: ../datos/procesados/yellow_tripdata_2025-01_clean.parquet
Retención de datos final: 93.52% (3,250,191 filas validas).


In [35]:
df = pd.read_parquet('../datos/procesados/yellow_tripdata_2025-01_clean.parquet')
print(df.head())

   VendorID tpep_pickup_datetime tpep_dropoff_datetime  passenger_count  \
0         1  2025-01-01 00:18:38   2025-01-01 00:26:59              1.0   
1         1  2025-01-01 00:32:40   2025-01-01 00:35:13              1.0   
2         1  2025-01-01 00:44:04   2025-01-01 00:46:01              1.0   
3         2  2025-01-01 00:14:27   2025-01-01 00:20:01              3.0   
4         2  2025-01-01 00:21:34   2025-01-01 00:25:06              3.0   

   trip_distance  RatecodeID store_and_fwd_flag  PULocationID  DOLocationID  \
0           1.60         1.0                  N           229           237   
1           0.50         1.0                  N           236           237   
2           0.60         1.0                  N           141           141   
3           0.52         1.0                  N           244           244   
4           0.66         1.0                  N           244           116   

   payment_type  ...  improvement_surcharge  total_amount  \
0            

In [36]:
print(df.info())

<class 'pandas.DataFrame'>
RangeIndex: 3250191 entries, 0 to 3250190
Data columns (total 25 columns):
 #   Column                 Dtype         
---  ------                 -----         
 0   VendorID               int32         
 1   tpep_pickup_datetime   datetime64[us]
 2   tpep_dropoff_datetime  datetime64[us]
 3   passenger_count        float64       
 4   trip_distance          float64       
 5   RatecodeID             float64       
 6   store_and_fwd_flag     str           
 7   PULocationID           int32         
 8   DOLocationID           int32         
 9   payment_type           int64         
 10  fare_amount            float64       
 11  extra                  float64       
 12  mta_tax                float64       
 13  tip_amount             float64       
 14  tolls_amount           float64       
 15  improvement_surcharge  float64       
 16  total_amount           float64       
 17  congestion_surcharge   float64       
 18  Airport_fee            float64   

In [37]:
print(df.isnull().sum())

VendorID                      0
tpep_pickup_datetime          0
tpep_dropoff_datetime         0
passenger_count               0
trip_distance                 0
RatecodeID                    0
store_and_fwd_flag       412924
PULocationID                  0
DOLocationID                  0
payment_type                  0
fare_amount                   0
extra                         0
mta_tax                       0
tip_amount                    0
tolls_amount                  0
improvement_surcharge         0
total_amount                  0
congestion_surcharge          0
Airport_fee                   0
cbd_congestion_fee            0
duracion_minutos              0
velocidad_mph                 0
dia_semana                    0
hora_dia                      0
es_fin_de_semana              0
dtype: int64


In [38]:
# Seleccionamos solo las columnas numéricas
columnas_numericas = df.select_dtypes(include=['number'])

# Contamos cuántos valores son menores a 0 en cada columna
conteo_negativos = (columnas_numericas < 0).sum()

# Filtramos para mostrar SOLO las columnas que tienen al menos 1 negativo
columnas_con_negativos = conteo_negativos[conteo_negativos > 0]

print("Columnas con valores negativos y su cantidad:")
display(columnas_con_negativos)


Columnas con valores negativos y su cantidad:


extra    101
dtype: int64

**Esto es de los extras que cobra el taxi en New York, y no es relevante para nuestro caso, por lo que quedará así